# BerryWorld: difficulty (D) sweep -- the Gate C test

The faithful-population run was a **clean null**: with enough training every condition solves poison avoidance equally (eat0 -> ~0.8) and enforcement goes extinct (selectivity decays to chance). Diagnosis: at `D=25` the environment signal alone is enough, so the taboo has nothing to add.

**This notebook cranks the difficulty.** The Koster effect can only exist when direct learning *fails*. Signature to look for: as `D` grows, `()` (no rule) plateaus **high** (can't solve it alone) while `(0,)` drops **below** it -- that gap is the effect.

Second axis: `r_zap_bonus` (a direct reward for zapping). D opens a niche; enforcement has to actually fill it, and we saw it go extinct at bonus=0. So we test whether a first-order incentive revives it where difficulty has made it matter.

**Run:** Runtime -> Change runtime type -> GPU. Then top to bottom; upload `berryworld.py`, `berryworld_jax.py`, `train_jax.py`, `run_sweep.py` when prompted.

In [ ]:
!pip install -q -U "jax[cuda12]" flax optax

In [ ]:
# upload the 4 code files
from google.colab import files
files.upload()

In [ ]:
# sanity: GPU + one tiny run
import jax; print('devices:', jax.devices())
import run_sweep as R
hp = dict(R.FAITHFUL_HP); hp['num_envs'] = 16; hp['updates'] = 5
_ = R.run_sweep([12], [(0,)], n_seeds=2, hp=hp,
                env_list=[R.env_variant(poison_delay=75)], out_csv='sanity.csv')

## 1. Difficulty sweep -- does higher D make `()` fail?
`D in {25,50,75}`, bonus 0, N=12. ~1.5e7 steps/cell. If OOM, lower `num_envs`/`n_seeds`.

In [ ]:
import run_sweep as R
hp = dict(R.FAITHFUL_HP); hp['num_envs'] = 64; hp['updates'] = 800
envs = [R.env_variant(poison_delay=d) for d in (25, 50, 75)]
rows = R.run_sweep([12], [(), (0,), (0, 1)], n_seeds=4,
                   hp=hp, env_list=envs, out_csv='dsweep.csv')

In [ ]:
# plot: eat0 (last-quarter) vs D, per condition -- the Koster signature
import csv, numpy as np, matplotlib.pyplot as plt
rows = list(csv.DictReader(open('dsweep.csv')))
Ds = sorted({int(r['D']) for r in rows})
conds = ['none', '0', '01']; labs = {'none':'() no rule','0':'(0,) important','01':'(0,1) silly'}
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
for c in conds:
    ys = []
    for D in Ds:
        sub = [r for r in rows if r['condition']==c and int(r['D'])==D]
        mx = max(int(r['update']) for r in sub)
        ys.append(np.mean([float(r['eat0']) for r in sub if int(r['update'])>=0.75*mx]))
    ax[0].plot(Ds, ys, 'o-', label=labs[c])
ax[0].set_xlabel('poison_delay D'); ax[0].set_ylabel('eat0 (last-quarter)')
ax[0].set_title('avoidance vs difficulty -- gap = Koster effect'); ax[0].legend()
# eat0 trajectories at the hardest D
Dh = Ds[-1]
for c in conds:
    sub = [r for r in rows if r['condition']==c and int(r['D'])==Dh]
    U = max(int(r['update']) for r in sub)+1
    y = np.array([[float(r['eat0']) for r in sub if int(r['update'])==u] for u in range(U)])
    ax[1].plot(y.mean(1), label=labs[c])
ax[1].set_title(f'eat0 trajectories at D={Dh}'); ax[1].set_xlabel('update'); ax[1].legend()
plt.tight_layout(); plt.savefig('dsweep.png', dpi=110); plt.show()
print('\nverdict (last-quarter eat0; gap of () above (0,) = effect):')
for D in Ds:
    r_ = {}
    for c in conds:
        sub = [r for r in rows if r['condition']==c and int(r['D'])==D]
        mx = max(int(r['update']) for r in sub)
        r_[c] = np.mean([float(r['eat0']) for r in sub if int(r['update'])>=0.75*mx])
    print(f"  D={D:2d}  ()={r_['none']:5.1f}  (0,)={r_['0']:5.1f}  (0,1)={r_['01']:5.1f}  gap ()-(0,)={r_['none']-r_['0']:+.1f}")

## 2. Enforcement incentive at the hardest D
If D=75 opened a niche but `(0,)` still doesn't separate, it's because enforcement is extinct. Test whether a direct zap reward revives it: `bonus in {0, 0.5}` at D=75.

In [ ]:
import run_sweep as R
hp = dict(R.FAITHFUL_HP); hp['num_envs'] = 64; hp['updates'] = 800
envs = [R.env_variant(poison_delay=75, r_zap_bonus=b) for b in (0.0, 0.5)]
rows2 = R.run_sweep([12], [(), (0,), (0, 1)], n_seeds=4,
                    hp=hp, env_list=envs, out_csv='dsweep_bonus.csv')
import csv, numpy as np
rows2 = list(csv.DictReader(open('dsweep_bonus.csv')))
print('\nD=75, does bonus revive enforcement?  (selectivity>1 = fired)')
for b in ('0.0','0.5'):
    for c in ('none','0','01'):
        sub = [r for r in rows2 if r['condition']==c and r['bonus']==b]
        if not sub: continue
        mx = max(int(r['update']) for r in sub)
        lq = [r for r in sub if int(r['update'])>=0.75*mx]
        sel = np.mean([float(r['selectivity']) for r in lq]); e0 = np.mean([float(r['eat0']) for r in lq])
        print(f'  bonus={b} {c:4s}  selectivity {sel:.2f}  eat0 {e0:.1f}')

In [ ]:
from google.colab import files
files.download('dsweep.csv')
# files.download('dsweep_bonus.csv')